## Step 1: Install Dependencies
Install Ultralytics so the YOLO training and validation APIs are available.


In [1]:
# Install the Ultralytics package used for YOLO11 training.
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.7 MB/s eta 0:00:00


## Step 2: Import Libraries
Load file, image, split, dataset download, PyTorch, and YOLO helpers.


In [ ]:
# Import file handling, image processing, dataset splitting, download, PyTorch, and YOLO tools.
import shutil
import yaml
import torch
import numpy as np
from pathlib import Path
 # Import the Image class from Pillow for opening, processing, and saving images
from PIL import Image
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import kagglehub
from google.colab import drive

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Step 3: Mount Drive and Download KITTI
Connect Google Drive, download the KITTI dataset, and store its local path.


In [ ]:
drive.mount('/content/drive')
klemenko_kitti_dataset_path = kagglehub.dataset_download('klemenko/kitti-dataset')

Mounted at /content/drive


100%|██████████| 22.5G/22.5G [03:58<00:00, 101MB/s]

Extracting files...


## Step 4: Configure Paths and Training Settings
Define dataset folders, classes, model settings, output paths, and compute device.


In [ ]:
# Set the base KITTI folder returned by KaggleHub.
KITTI_BASE_DIR = klemenko_kitti_dataset_path

# Define source image and label folders inside the KITTI dataset.
IMAGE_DIR = Path(KITTI_BASE_DIR) / 'data_object_image_2' / 'training' / 'image_2'
LABEL_DIR = Path(KITTI_BASE_DIR) / 'data_object_label_2' / 'training' / 'label_2'

# Define local YOLO output folders for train, validation, and converted labels.
TRAIN_DIR = Path('train')
VALID_DIR = Path('valid')
LABELS_DIR = Path('labels_with_dont_care')

# Keep the KITTI classes that this model should learn.
CLASSES = [
    'Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting'
]

# Define model, training settings, output folder, and validation threshold.
MODEL_ARCH = 'yolo11n.pt'
EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 640
# Confidence threshold for validation and inference (lowered to 0.25 to capture more detections, but can be adjusted based on results)
CONFIDENCE_THRESHOLD = 0.25
PROJECT_NAME = '/content/drive/MyDrive/YOLO11-KITTI'
EXPERIMENT_NAME = 'exp1'

# Use GPU when available; otherwise fall back to CPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Stop early if the expected KITTI image or label folders are missing.
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Image directory not found: {IMAGE_DIR}")
if not LABEL_DIR.exists():
    raise FileNotFoundError(f"Label directory not found: {LABEL_DIR}")

Using device: cuda


## Step 5: Prepare Label Conversion Helpers
Create class-index mapping and helper functions to convert KITTI boxes into YOLO labels.


In [ ]:
# Map class names to YOLO numeric class IDs.
CLAZZ_NUMBERS = {name: idx for idx, name in enumerate(CLASSES)}


# KITTI labels store boxes with pixel corners: left, top, right, bottom.
# YOLO needs boxes as x_center, y_center, width, height instead of corners.
# YOLO also needs those values normalized from 0 to 1, so they work for any image size.
# This function converts KITTI's pixel-corner box into YOLO's normalized center box.
def convert_bbox_to_yolo(bbox, size):
    """Convert KITTI box coordinates to normalized YOLO format."""
    # bbox is ordered as left, right, top, bottom after reading the KITTI row.
    # size is ordered as image width, image height from PIL.Image.size.
    dw = 1.0 / size[0]  # Divide x values by image width to normalize them.
    dh = 1.0 / size[1]  # Divide y values by image height to normalize them.

    # Find the horizontal center between the left and right edges.
    x_center = (bbox[0] + bbox[1]) / 2.0

    # Find the vertical center between the top and bottom edges.
    y_center = (bbox[2] + bbox[3]) / 2.0

    # Calculate the box width in pixels.
    width = bbox[1] - bbox[0]

    # Calculate the box height in pixels.
    height = bbox[3] - bbox[2]

    # Normalize x center and width using the image width.
    x_center *= dw
    width *= dw

    # Normalize y center and height using the image height.
    y_center *= dh
    height *= dh

    # Return values in YOLO order: x_center, y_center, width, height.
    return x_center, y_center, width, height

# KITTI stores one text file per image, with one object per line.
# Each KITTI line includes class name, 3D fields, and a 2D pixel box.
# YOLO needs one label row per object: class_id x_center y_center width height.
# This function reads a KITTI file and builds the YOLO rows for one image.
def parse_kitti_label_file(lbl_path, img_path):
    """Read one KITTI label file and return YOLO label rows."""
    # Open the KITTI label file and split it into separate object lines.
    with open(lbl_path, 'r', encoding='utf-8') as file:
        lines = file.read().strip().split('\n')

    # Start with an empty list; each item added will be one YOLO label row.
    yolo_labels = []

    # If the matching image is missing, we cannot normalize the box sizes.
    if not img_path.exists():
        return yolo_labels

    # Read image width and height; YOLO normalization depends on image size.
    img_size = Image.open(img_path).size

    # Process each object annotation from the KITTI file.
    for line in lines:
        parts = line.split()

        # The first column is the KITTI class name, like Car or Pedestrian.
        clazz = parts[0]

        # Skip classes we did not include in the training class list.
        if clazz not in CLAZZ_NUMBERS:
            continue

        # KITTI 2D bbox columns are: left=4, top=5, right=6, bottom=7.
        bbox_left = float(parts[4])
        bbox_top = float(parts[5])
        bbox_right = float(parts[6])
        bbox_bottom = float(parts[7])

        # Reorder the box to match convert_bbox_to_yolo: left, right, top, bottom.
        bbox = (bbox_left, bbox_right, bbox_top, bbox_bottom)

        # Convert the KITTI pixel-corner box into YOLO normalized center format.
        x_center, y_center, width, height = convert_bbox_to_yolo(bbox, img_size)

        # Convert class name into the numeric class ID YOLO expects.
        clazz_number = CLAZZ_NUMBERS[clazz]

        # Save the final YOLO row: class_id x_center y_center width height.
        yolo_labels.append((clazz_number, x_center, y_center, width, height))

    return yolo_labels


## Step 6: Generate YOLO Labels
Read KITTI label files and write YOLO-formatted labels for every matching image.


In [6]:
# Create the folder that will contain converted YOLO label files.
if not LABELS_DIR.exists():
    LABELS_DIR.mkdir()

# Collect KITTI image and label paths.
image_paths = sorted(list(IMAGE_DIR.glob('*.png')))
label_paths = sorted(list(LABEL_DIR.glob('*.txt')))

# Convert each image's KITTI label file into a YOLO label file.
for img_path in image_paths:
    lbl_path = LABEL_DIR / f"{img_path.stem}.txt"
    if lbl_path.exists():
        yolo_labels = parse_kitti_label_file(lbl_path, img_path)
        yolo_label_path = LABELS_DIR / f"{img_path.stem}.txt"
        with open(yolo_label_path, 'w', encoding='utf-8') as lf:
            for lbl in yolo_labels:
                lf.write(" ".join(f"{val:.6f}" for val in lbl) + "\n")

print("YOLO format labels have been generated in:", LABELS_DIR.resolve())

YOLO format labels have been generated in: /content/labels_with_dont_care


## Step 7: Split Dataset
Create train/validation pairs, rebuild YOLO folder structure, and copy each image-label pair.


In [ ]:
# Keep only images that have converted YOLO labels.
labels_for_images = [(img_path, LABELS_DIR / f"{img_path.stem}.txt")
                     for img_path in image_paths
                     if (LABELS_DIR / f"{img_path.stem}.txt").exists()]

# Split image-label pairs into training and validation sets.
train_pairs, valid_pairs = train_test_split(
    labels_for_images,
    test_size=0.1,
    random_state=42,
    shuffle=True
)
print(f"Training samples: {len(train_pairs)}, Validation samples: {len(valid_pairs)}")

# Rebuild the YOLO train and validation folder structure.
# train
#    images
#    labels
# valid
#    images
#    labels
for folder in [TRAIN_DIR, VALID_DIR]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir()
    (folder / 'images').mkdir()
    (folder / 'labels').mkdir()

# Copy training image-label pairs into YOLO folders.
for img_path, lbl_path in train_pairs:
    shutil.copy(img_path, TRAIN_DIR / 'images' / img_path.name)
    shutil.copy(lbl_path, TRAIN_DIR / 'labels' / lbl_path.name)

# Copy validation image-label pairs into YOLO folders.
for img_path, lbl_path in valid_pairs:
    shutil.copy(img_path, VALID_DIR / 'images' / img_path.name)
    shutil.copy(lbl_path, VALID_DIR / 'labels' / lbl_path.name)

print(f"Training data copied to {TRAIN_DIR / 'images'} and {TRAIN_DIR / 'labels'}")
print(f"Validation data copied to {VALID_DIR / 'images'} and {VALID_DIR / 'labels'}")

Training samples: 6732, Validation samples: 749
Training data copied to train/images and train/labels
Validation data copied to valid/images and valid/labels


## Step 8: Create YOLO Data Config
Write `data.yaml` with train/validation image paths and class metadata.


In [8]:
# Build the YOLO data configuration with split paths and class names.
DATA_CONFIG = 'data.yaml'
data_config = {
    'train': str((TRAIN_DIR / 'images').resolve()),
    'val': str((VALID_DIR / 'images').resolve()),
    'names': CLASSES,
    'nc': len(CLASSES)
}

# Save data.yaml for the Ultralytics trainer.
with open(DATA_CONFIG, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("data.yaml file created with content:")
print(data_config)

data.yaml file created with content:
{'train': '/content/train/images', 'val': '/content/valid/images', 'names': ['Car', 'Van', 'Truck', 'Pedestrian', 'Person_sitting'], 'nc': 5}


## Step 9: Train YOLO11 KITTI Model
Load YOLO11n and fine-tune it on the prepared KITTI dataset.


In [ ]:
# Load the pretrained model on the selected device.
model = YOLO(MODEL_ARCH).to(device)

# Train YOLO11 on the prepared KITTI dataset.
train_results = model.train(
    data=DATA_CONFIG,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    project=PROJECT_NAME,
    name=EXPERIMENT_NAME,
    device=device,
    exist_ok=True,
    patience=15,        # stops if no improvement for 15 epochs
    lr0=0.001,          # lower LR for fine-tuning
    lrf=0.01,           # final LR ratio
)

print("\nTraining completed!\n")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective=0.0, plots=

## Step 10: Validate Best Weights
Load the best checkpoint, run validation, and print available detection metrics.


In [ ]:
# Load the best checkpoint from the training run.
best_weights_path = f'{PROJECT_NAME}/{EXPERIMENT_NAME}/weights/best.pt'
model = YOLO(best_weights_path).to(device)

# Run validation on the validation split.
validation_results = model.val(
    data=DATA_CONFIG,
    split='val',
    conf=CONFIDENCE_THRESHOLD,
    save=False,
    plots=False
)
print("\nValidation completed!\n")

# Print the raw validation object and public attributes for inspection.
print("Validation Results (raw DetMetrics object):")
print(validation_results)

print("Attributes of validation_results:")
for attribute in dir(validation_results):
    if not attribute.startswith('_'):
        print(attribute, "=", getattr(validation_results, attribute))

# Extract common box metrics from the validation result.
metrics = validation_results.box
print("\nBox Metrics:")
print(metrics)

precision = getattr(metrics, 'mp', None)
recall = getattr(metrics, 'mr', None)

# Calculate F1 score when precision and recall are both available.
f1_score = None
if precision is not None and recall is not None and (precision + recall) > 0:
    f1_score = 2 * (precision * recall) / (precision + recall)

# Accuracy and confusion matrix are placeholders for object detection reporting.
accuracy = None
confusion_matrix = None

print("\nConfusion Matrix:")
if confusion_matrix is not None:
    print(confusion_matrix)
else:
    print("[[ ... ]]")

if accuracy is not None:
    print(f"Accuracy: {accuracy * 100:.2f}%")
else:
    print("Accuracy: Not Applicable for object detection")

if precision is not None:
    print(f"Precision: {precision:.2f}")
else:
    print("Precision: Not Available")

if recall is not None:
    print(f"Recall: {recall:.2f}")
else:
    print("Recall: Not Available")

if f1_score is not None:
    print(f"F1 Score: {f1_score:.2f}")
else:
    print("F1 Score: Not Available")

## Step 11: Predict on Validation Images
Run predictions on validation images and save the rendered detections.


In [ ]:
# Run predictions on validation images and save annotated outputs.
val_predictions = model.predict(
    source=str((VALID_DIR / 'images').resolve()),
    save=True,
    conf=CONFIDENCE_THRESHOLD
)

# Print where prediction images were saved, if any predictions ran.
if val_predictions:
    predictions_save_dir = val_predictions[0].save_dir
    print(f"\nPredictions saved to '{predictions_save_dir}'.\n")
else:
    print("No predictions were made.")